# Caso I · 04 Comparativa pandas vs Spark — cuándo merece la pena

> _Tutorial · Caso de uso: **I — Spark vs Pandas** · Capa Medallion: **oro** · Spec: `docs/specs/synthetic-bms/01-product-spec.md`_

Material docente del proyecto **CAPTIA Synthetic Data BMS** — IES Dr. Lluís Simarro,
Curso de Especialización IA & Big Data 2025-2026.


## 1. Objetivo

Combinar los resultados de los notebooks 02 y 03 y emitir una recomendación clara.


## 2. Qué se aprende

- Speedup en función del tamaño.
- Punto de cruce pandas / Spark.
- Coste fijo de Spark.


## 3. Contexto del caso de uso

Recomendación final del proyecto.


## 4. Relación con CENTINELA+

Tomas de decisión.


## 5. Relación con Medallion

Oro: análisis.


## 6. Datos de entrada

JSON tiempos.


## 7. Schema CAPTIA esperado

No aplica.


## 8. Setup y variables de entorno

Cargamos las variables de entorno (`.env`), inicializamos `numpy` con `seed=42` y aplicamos el estilo de plotting compartido. Los helpers viven en `notebooks/_common/`.


In [ ]:
# Setup canónico — todos los notebooks didácticos lo usan
from __future__ import annotations

import sys
from pathlib import Path

ROOT = Path.cwd()
while ROOT.name and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from notebooks._common.captia_schema import (
    CANONICAL_TAGS, MEASUREMENT_TELEMETRY, MEASUREMENT_FAULT_LABELS,
    DEFAULT_BUCKET_RETENTIONS, KNOWN_VARIABLES,
    build_topic, build_line_protocol, validate_canonical_tags,
)
from notebooks._common.connection import load_env, get_influx_client, get_default_bucket
from notebooks._common.plotting import setup_default_style, plot_timeseries, plot_distribution
import notebooks._common.synthetic_mocks as mocks

SEED = 42
rng = np.random.default_rng(SEED)
setup_default_style()
load_env()
print(f"ROOT={ROOT}, SEED={SEED}, default_bucket={get_default_bucket()}")


## 9. Carga de datos o mock

Combinamos resultados de demo.


In [ ]:
demo_pd = pd.DataFrame([{"op": "groupby_building", "median_s": 0.05},
                          {"op": "groupby_hour_dow", "median_s": 0.07}])
demo_sp = pd.DataFrame([{"op": "groupby_building", "spark_s": 1.2},
                          {"op": "groupby_hour_dow", "spark_s": 1.5}])
combined = demo_pd.merge(demo_sp, on="op")
combined["speedup_pandas_better"] = combined["spark_s"] / combined["median_s"]
combined


## 10. Exploración paso a paso

En subsets pequeños pandas gana.


In [ ]:
sizes = [1e3, 1e4, 1e5, 1e6, 1e7]
# Modelo simplificado: pandas O(N), Spark = startup_const + alpha*N (alpha << pandas alpha cuando N grande)
pd_t = [n * 1e-7 for n in sizes]
sp_t = [1.0 + n * 1e-9 for n in sizes]
table = pd.DataFrame({"N": sizes, "pandas_s": pd_t, "spark_s": sp_t})
table["winner"] = np.where(np.array(pd_t) < np.array(sp_t), "pandas", "spark")
table


## 11. Transformación bronce → plata

No aplica.


## 12. Construcción de capa oro

Plot speedup.


In [ ]:
plt.figure(figsize=(8, 3))
plt.plot(sizes, pd_t, label="pandas", color="#3F51B5", marker="o")
plt.plot(sizes, sp_t, label="spark", color="#FF5722", marker="o")
plt.xscale("log"); plt.yscale("log")
plt.xlabel("N filas"); plt.ylabel("segundos")
plt.legend(); plt.title("Cruce pandas vs Spark (modelo)")
plt.tight_layout()


## 13. Visualizaciones explicativas

Recomendación textual.


In [ ]:
crossover = sizes[next((i for i, (p, s) in enumerate(zip(pd_t, sp_t)) if p > s), -1)]
print(f"Punto de cruce aproximado: {crossover:.0e} filas")
print("Recomendación: Spark cuando el dataset > 1M filas o cuando se proyecte crecer.")


## 14. Validaciones

El cruce está en escala log.


In [ ]:
assert any(p > s for p, s in zip(pd_t, sp_t))


## 15. Errores comunes

1. Recomendar Spark sin medir.
2. No considerar coste operativo del cluster.
3. Comparar Spark single-node con pandas — no es justo.


## 16. Ejercicios propuestos

1. Mide el modelo real con tu subset.
2. Calcula el tamaño BDG2 completo y predice speedup.
3. Ensaya `polars` como alternativa.


## 17. Cómo se reutiliza con datos reales

Re-ejecutar con BDG2 completo en cluster ITI.


## 18. Resumen final y próximos pasos

Recuerda los conceptos principales del notebook y enlaza al siguiente paso.

- Siguiente notebook: `10_case_J_traffic_yolo/01_captura_imagenes_dgt.ipynb`.
- Documento web del caso: `docs/use-cases/case-i-spark-pandas.md`.
